# Experiment dashboard

Loads runs from W&B and renders them with **matplotlib**, driven by ipywidgets.

**Why this is a notebook and not a W&B workspace.** W&B custom charts are Vega-Lite specs. They
cannot render matplotlib, cannot drive a slider over run configs, and cannot build a grid whose
columns you pick from a dropdown with per-column legends. So W&B stays the data store and the
plotting happens here, reusing the `interact` + `FloatSlider` pattern already in the old
notebook.

Three panels:
1. **Grid** — pick any logged metrics into a 2-column grid, one legend per column.
2. **Sweep** — slider + Play animation over a swept axis, plus mp4 export.
3. **Compare** — overlay one metric across runs, filtered by any config key.

## Setup

In [ ]:
import os, sys, pathlib, functools
sys.path.insert(1, os.path.abspath(os.path.join('..', 'src')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation
import wandb

%matplotlib inline

from config import load_config, get_in

# Everything configurable lives in configs/, never in this notebook.
CFG     = load_config("base.yaml")
ENTITY  = get_in(CFG, "wandb.entity")
PROJECT = get_in(CFG, "wandb.project")
CACHE   = pathlib.Path("..") / get_in(CFG, "paths.cache")
CACHE.mkdir(parents=True, exist_ok=True)
print(f"{ENTITY}/{PROJECT}  cache -> {CACHE}")

## Loading

`run.history()` is **downsampled by default** — it would silently smooth exactly the catapult
spikes this project studies. `scan_history()` returns every logged step; it is slower, so the
result is cached to parquet and the download happens once per run.

In [ ]:
def load_runs(group=None, filters=None, force=False):
    '''Fetch runs from W&B -> {run_name: (config dict, DataFrame)}. Cached to parquet.'''
    api = wandb.Api()
    f = dict(filters or {})
    if group:
        f["group"] = group
    runs = api.runs(f"{ENTITY}/{PROJECT}", filters=f or None)
    out = {}
    for r in runs:
        cache = CACHE / f"{r.id}.parquet"
        if cache.exists() and not force:
            df = pd.read_parquet(cache)
        else:
            df = pd.DataFrame(list(r.scan_history()))   # exact, not downsampled
            if len(df):
                df.to_parquet(cache)
        if len(df):
            out[r.name] = (dict(r.config), df)
    print(f"loaded {len(out)} runs" + (f" from group '{group}'" if group else ""))
    return out


def metric_keys(runs):
    '''Every numeric column present in any run, minus W&B bookkeeping.'''
    keys = set()
    for _, df in runs.values():
        keys |= {c for c in df.columns if not c.startswith("_") and
                 pd.api.types.is_numeric_dtype(df[c])}
    return sorted(keys)


def config_keys(runs):
    '''Config keys that actually vary across the loaded runs -- the useful filters.'''
    if not runs: return []
    allk = set().union(*[set(c) for c, _ in runs.values()])
    return sorted(k for k in allk
                  if len({str(c.get(k)) for c, _ in runs.values()}) > 1)

In [ ]:
# RUNS = load_runs(group="lr_sweep")
RUNS = load_runs()          # everything in the project
print(f"{len(metric_keys(RUNS))} metrics, varying config keys: {config_keys(RUNS)[:8]}")

## Panel 1 — metric grid

Two multi-selects, one per column. Each column draws its chosen metrics on stacked axes with its
own legend, matching the 4x2 layout of the old notebook's cell 52.

In [ ]:
COLORS = dict(loss='steelblue', spectral='seagreen', hessian='crimson',
              nfa='darkorange', cov='mediumpurple', sparsity='sienna')
PALETTE = list(COLORS.values()) + ['k', 'tab:pink', 'tab:olive', 'tab:cyan']


def plot_grid(run, left, right, logy_left=True, logy_right=False, figsize=(14, 8)):
    '''One run, two columns of stacked axes, a legend per column.'''
    cfg, df = run
    rows = max(len(left), len(right), 1)
    fig, axs = plt.subplots(rows, 2, figsize=figsize, squeeze=False)
    for col, keys, logy in ((0, left, logy_left), (1, right, logy_right)):
        for row in range(rows):
            ax = axs[row][col]
            if row < len(keys):
                k = keys[row]
                ax.plot(df[k].values, label=k, color=PALETTE[(row + 3 * col) % len(PALETTE)])
                ax.set(ylabel=k.split('/')[-1], yscale='log' if logy else 'linear')
                if row == rows - 1: ax.set_xlabel('step')
                ax.grid(alpha=.3)
            else:
                ax.axis('off')
    for col in (0, 1):
        h, l = [], []
        for ax in axs[:, col]:
            hh, ll = ax.get_legend_handles_labels(); h += hh; l += ll
        if h:
            x = (axs[0, col].get_position().x0 + axs[0, col].get_position().x1) / 2
            fig.legend(h, l, loc='upper center', bbox_to_anchor=(x, 1.0),
                       bbox_transform=fig.transFigure, framealpha=1.0, fontsize=8)
    plt.tight_layout(); fig.subplots_adjust(top=0.80)
    return fig


def grid_panel(runs):
    keys = metric_keys(runs)
    run_dd = W.Dropdown(options=sorted(runs), description='run:', layout=W.Layout(width='70%'))
    left   = W.SelectMultiple(options=keys, description='left:',  rows=10,
                              value=tuple(k for k in keys if k in ('loss',)))
    right  = W.SelectMultiple(options=keys, description='right:', rows=10,
                              value=tuple(k for k in keys if 'nfa' in k)[:3])
    lg = W.Checkbox(value=True,  description='log left')
    rg = W.Checkbox(value=False, description='log right')
    outw = W.Output()

    def draw(*_):
        with outw:
            outw.clear_output(wait=True)
            plot_grid(runs[run_dd.value], list(left.value), list(right.value),
                      lg.value, rg.value)
            plt.show()
    for w in (run_dd, left, right, lg, rg): w.observe(draw, names='value')
    draw()
    return W.VBox([run_dd, W.HBox([left, right]), W.HBox([lg, rg]), outw])

grid_panel(RUNS)

## Panel 2 — sweep slider and animation

Replaces the old `plot_for_lr` + `FloatSlider`. The axis is any config key that varies, so the
same panel works for learning-rate sweeps, parametrization sweeps and width sweeps.

In [ ]:
def sweep_panel(runs, default_axis='lr_absolute'):
    ckeys = config_keys(runs) or ['lr_absolute']
    axis  = W.Dropdown(options=ckeys, value=default_axis if default_axis in ckeys else ckeys[0],
                       description='axis:')
    metric = W.Dropdown(options=metric_keys(runs), value='loss', description='metric:')
    idx = W.IntSlider(min=0, max=max(len(runs) - 1, 0), description='run:', continuous_update=False)
    play = W.Play(min=0, max=max(len(runs) - 1, 0), interval=400)
    W.jslink((play, 'value'), (idx, 'value'))
    logy = W.Checkbox(value=True, description='log y')
    mk = W.Button(description='render mp4', button_style='info')
    outw = W.Output()

    def ordered():
        items = [(c.get(axis.value), nm, df) for nm, (c, df) in runs.items()]
        return sorted(items, key=lambda t: (t[0] is None, t[0]))

    def draw(*_):
        items = ordered()
        idx.max = max(len(items) - 1, 0)
        play.max = idx.max
        val, nm, df = items[min(idx.value, len(items) - 1)]
        with outw:
            outw.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(df[metric.value].values, label=f"{axis.value}={val}")
            lo = min(d[metric.value].min() for _, _, d in items if metric.value in d)
            hi = max(d[metric.value].max() for _, _, d in items if metric.value in d)
            ax.set(xlabel='step', ylabel=metric.value,
                   yscale='log' if logy.value else 'linear', title=nm, ylim=(lo, hi))
            ax.legend(); ax.grid(alpha=.3); plt.show()

    def render(_):
        items = ordered()
        with outw:
            outw.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(12, 5))
            line, = ax.plot(items[0][2][metric.value].values)
            lo = min(d[metric.value].min() for _, _, d in items if metric.value in d)
            hi = max(d[metric.value].max() for _, _, d in items if metric.value in d)
            ax.set(xlabel='step', ylabel=metric.value,
                   yscale='log' if logy.value else 'linear', ylim=(lo, hi))
            ttl = ax.set_title("")
            def upd(i):
                v, nm, df = items[i]
                line.set_ydata(df[metric.value].values)
                ttl.set_text(f"{axis.value}={v}")
                return line, ttl
            anim = FuncAnimation(fig, upd, frames=len(items), interval=300, blit=False)
            display(HTML(anim.to_html5_video()))   # needs: module load ffmpeg
            plt.close(fig)

    for w in (axis, metric, idx, logy): w.observe(draw, names='value')
    mk.on_click(render)
    draw()
    return W.VBox([W.HBox([axis, metric, logy]), W.HBox([play, idx, mk]), outw])

sweep_panel(RUNS)

## Panel 3 — overlay across runs

One metric, every run that matches a config filter, on a single axis.

In [ ]:
def compare_panel(runs):
    metric = W.Dropdown(options=metric_keys(runs), value='loss', description='metric:')
    ckeys  = ['(all)'] + config_keys(runs)
    fkey   = W.Dropdown(options=ckeys, value='(all)', description='filter:')
    fval   = W.Dropdown(options=['(all)'], description='value:')
    logy   = W.Checkbox(value=True, description='log y')
    outw   = W.Output()

    def refresh_vals(*_):
        if fkey.value == '(all)':
            fval.options = ['(all)']
        else:
            fval.options = ['(all)'] + sorted({str(c.get(fkey.value)) for c, _ in runs.values()})
        fval.value = '(all)'

    def draw(*_):
        with outw:
            outw.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(12, 5))
            n = 0
            for nm, (c, df) in sorted(runs.items()):
                if fkey.value != '(all)' and fval.value != '(all)' \
                   and str(c.get(fkey.value)) != fval.value:
                    continue
                if metric.value in df:
                    ax.plot(df[metric.value].values, label=nm, lw=1.2); n += 1
            ax.set(xlabel='step', ylabel=metric.value,
                   yscale='log' if logy.value else 'linear')
            if n: ax.legend(fontsize=6, ncol=2)
            ax.grid(alpha=.3); plt.show()

    fkey.observe(lambda *_: (refresh_vals(), draw()), names='value')
    for w in (metric, fval, logy): w.observe(draw, names='value')
    refresh_vals(); draw()
    return W.VBox([W.HBox([metric, logy]), W.HBox([fkey, fval]), outw])

compare_panel(RUNS)